# ATL_ICE_2A Processing — v2.0

Grids **ice water content (IWC)** and **effective radius (r_eff)** from the ATL_ICE_2A
product onto a global 1° × 100 m grid for December 2025.

## What is new in v2 compared to v1

| | v1 | v2 |
|---|---|---|
| TC orbit matching | orbit-key dict (may silently miss matches) | time-overlap matching (same as diagnostic) |
| Fill-value filter | none (`np.isfinite` only) | `d > 0` — rejects ATL_ICE fill zeros |
| Rejection counters | none | printed per batch for diagnostics |

The v1 output was contaminated by ATL_ICE_2A fill values (0.0), which passed
`np.isfinite` and pulled cell means toward zero.  The `d > 0` filter in v2 removes them.

## Workflow
1. Run **Cells 0–4** to set up (imports, helpers, config, discovery, grid).
2. Run **Cell 5** to define the core functions.
3. Run **Cell 6** (diagnostic test, 5 ZIPs) and review the printed stats.
   If IWC and r_eff means look physically plausible, proceed.
4. Run **Cells 7–9** to process the full batch and save output.

**Runs on:** Work PC (needs access to remote ICE and TC data).

## 0. Imports

In [ ]:
import earthcarekit as eck
from pathlib import Path

_cfg = Path("/usr/people/raucher/Documents/Config_ECK/default_config.toml")
if _cfg.exists():
    eck.set_config(str(_cfg))

import xarray as xr
import numpy as np
import re as _re
import sys, shlex, shutil
from datetime import datetime, timedelta, date
from scipy.interpolate import interp1d

MY_SUBPROCESS_FILE = Path("/usr/people/raucher/Documents/Coding1/Gerd-Jan/OneDrive_1_24-2-2026")
if str(MY_SUBPROCESS_FILE) not in sys.path:
    sys.path.insert(0, str(MY_SUBPROCESS_FILE))
from my_subprocess import run_shell_cmd_and_communicate, print_shell_output

print("Imports OK")

## 0b. ZIP helpers

The raw EarthCARE data is stored on the remote server as one ZIP per orbit, each
containing one HDF5 file.  We copy one ZIP at a time to local scratch, process it,
then delete it — never modifying remote files.

ATL_ICE_2A and ATL_TC__2A use **different orbit numbering**, so TC files are matched
by time overlap rather than orbit key.

In [ ]:
def _to_date(v):
    if isinstance(v, datetime): return v.date()
    if isinstance(v, date):     return v
    if isinstance(v, str):      return datetime.strptime(v, "%Y-%m-%d").date()
    raise TypeError(type(v))


def run_cmd_checked(cmd):
    lines_out, lines_err, rc = run_shell_cmd_and_communicate(cmd, verbose=False)
    if rc != 0:
        print_shell_output(lines_out, lines_err, prefix="[shell] ")
        raise RuntimeError(f"Command failed (exit {rc}): {cmd}")
    return lines_out, lines_err


def discover_remote_zip_files(root, start_date, end_date):
    root, start, end = Path(root), _to_date(start_date), _to_date(end_date)
    zips, day = [], start
    while day <= end:
        d = root / day.strftime("%Y") / day.strftime("%m") / day.strftime("%d")
        if d.exists():
            zips.extend(sorted(d.glob("*.ZIP")))
            zips.extend(sorted(d.glob("*.zip")))
        day += timedelta(days=1)
    return sorted(dict.fromkeys(zips))


def stage_zip_and_extract(src_zip, stage_root):
    stage_root.mkdir(parents=True, exist_ok=True)
    local_zip   = stage_root / src_zip.name
    extract_dir = stage_root / src_zip.stem
    if local_zip.exists():   local_zip.unlink()
    if extract_dir.exists(): shutil.rmtree(extract_dir)
    run_cmd_checked(f"cp {shlex.quote(str(src_zip))} {shlex.quote(str(local_zip))}")
    run_cmd_checked(f"unzip -oq {shlex.quote(str(local_zip))} -d {shlex.quote(str(extract_dir))}")
    h5_files = sorted(extract_dir.rglob("*.h5"))
    if not h5_files:
        raise FileNotFoundError(f"No .h5 files found after extracting: {src_zip}")
    return local_zip, extract_dir, h5_files


def cleanup_staged_data(local_zip, extract_dir):
    if extract_dir is not None and extract_dir.exists(): shutil.rmtree(extract_dir)
    if local_zip   is not None and local_zip.exists():   local_zip.unlink()


_DT_FMT = "%Y%m%dT%H%M%SZ"

def _parse_times(p):
    m = _re.search(r'_(\d{8}T\d{6}Z)_(\d{8}T\d{6}Z)_', Path(p).name)
    if not m: return None, None
    return datetime.strptime(m.group(1), _DT_FMT), datetime.strptime(m.group(2), _DT_FMT)


def build_tc_time_index(tc_zips):
    index = []
    for p in tc_zips:
        s, e = _parse_times(p)
        if s is not None:
            index.append((s, e, p))
    return index


def find_matching_tc_zip(ice_zip, tc_index):
    """Return the TC ZIP with the greatest time overlap with ice_zip."""
    s1, e1 = _parse_times(ice_zip)
    if s1 is None: return None
    best_path, best_ov = None, timedelta(0)
    for s2, e2, path in tc_index:
        ov = min(e1, e2) - max(s1, s2)
        if ov > best_ov:
            best_ov, best_path = ov, path
    return best_path


print("ZIP helpers defined")

## 1. Config

In [ ]:
# ── Remote data paths ─────────────────────────────────────────────────────────
REMOTE_ICE_ROOT = Path("/net/pc230016/nobackup_1/users/zadelhof/EarthCARE_DATA/L2/ATL_ICE_2A")
REMOTE_TC_ROOT  = Path("/net/pc230016/nobackup_1/users/zadelhof/EarthCARE_DATA/L2/ATL_TC__2A")

# ── Local scratch ─────────────────────────────────────────────────────────────
LOCAL_STAGE_ICE = Path("/usr/people/raucher/Documents/Coding1/KNMI/KNMI/temp_ZIPs_ICE")
LOCAL_STAGE_TC  = Path("/usr/people/raucher/Documents/Coding1/KNMI/KNMI/temp_ZIPs_TC_for_ICE")
LOCAL_STAGE_ICE.mkdir(parents=True, exist_ok=True)
LOCAL_STAGE_TC.mkdir(parents=True, exist_ok=True)

# ── Output directory ──────────────────────────────────────────────────────────
OUTPUT_DIR = Path("/usr/people/raucher/Documents/Coding1/KNMI/KNMI/ATC_output/20251201_20251231")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Date range ────────────────────────────────────────────────────────────────
START_DATE = "2025-12-01"
END_DATE   = "2025-12-31"

# ── Variables ─────────────────────────────────────────────────────────────────
# HDF5 variable name → short name used in output NetCDF
# Units: ice_effective_radius in 10⁻⁶ m (μm), ice_water_content in 10⁻⁶ kg m⁻³ (mg m⁻³)
DATA_VARS = {
    "ice_effective_radius": "eff_radius",
    "ice_water_content":    "iwc",
}

# ── Fill-value filter ─────────────────────────────────────────────────────────
# ATL_ICE_2A stores 0.0 as a fill value for height bins where the retrieval did
# not converge.  np.isfinite(0.0) is True, so without this filter, fill values
# pass through and pull cell means toward zero.
# Any pixel with value <= 0 is treated as a failed retrieval and discarded.
FILL_VALUE_THRESHOLD = 0.0   # reject pixels with value <= this

# ── ATL_TC ice mask settings ──────────────────────────────────────────────────
TC_CLASS_VAR     = "classification"
TC_ICE_CODE      = 3
TC_EXCLUDE_CODES = [-2, -1, -3]
TC_QC_VAR        = "quality_status"
TC_MAX_QC_FLAG   = 1
HEIGHT_VAR       = "height"

# ── Grid settings (must match all other v2 notebooks) ────────────────────────
GRID_RES_DEG = 1.0
MAX_HEIGHT_M = 20_000.0
HEIGHT_STEP  = 100.0
MIN_SAMPLES  = 10

lat_bins = np.arange(-90.0,  90.0  + GRID_RES_DEG, GRID_RES_DEG)
lon_bins = np.arange(-180.0, 180.0 + GRID_RES_DEG, GRID_RES_DEG)
target_h = np.arange(0.0, MAX_HEIGHT_M + 1.0, HEIGHT_STEP)

n_lat = len(lat_bins) - 1
n_lon = len(lon_bins) - 1
n_h   = len(target_h)

start_dt = datetime.strptime(START_DATE, "%Y-%m-%d").date()
end_dt   = datetime.strptime(END_DATE,   "%Y-%m-%d").date()

print("Config ready")
print(f"Date range : {start_dt} → {end_dt}")
print(f"Grid       : {n_lat} lat × {n_lon} lon × {n_h} height levels")
print(f"Variables  : {list(DATA_VARS.values())}")

## 2. File discovery

ATL_ICE_2A and ATL_TC__2A use different orbit numbering, so TC files are matched
to ICE files by time overlap (same approach as the diagnostic histogram notebook).

In [ ]:
ice_zips = discover_remote_zip_files(REMOTE_ICE_ROOT, START_DATE, END_DATE)
tc_zips  = discover_remote_zip_files(REMOTE_TC_ROOT,  START_DATE, END_DATE)

tc_index = build_tc_time_index(tc_zips)
tc_zip_for_ice = {z: find_matching_tc_zip(z, tc_index) for z in ice_zips}

n_matched   = sum(1 for v in tc_zip_for_ice.values() if v is not None)
n_unmatched = len(ice_zips) - n_matched

print(f"ICE ZIPs found : {len(ice_zips)}")
print(f"TC  ZIPs found : {len(tc_zips)}")
print(f"Matched        : {n_matched}")
if n_unmatched:
    print(f"WARNING: {n_unmatched} ICE ZIPs have no matching TC file and will be skipped.")

print(f"\nExample match:")
print(f"  ICE: {ice_zips[0].name}")
print(f"  TC : {tc_zip_for_ice[ice_zips[0]].name if tc_zip_for_ice[ice_zips[0]] else 'None'}")

## 3. Single-file inspection

Open the first ICE file to verify the data structure and confirm that fill zeros
are present (justifying the `d > 0` filter).  **Run before the full batch.**

In [ ]:
_local_zip, _extract_dir, _h5s = stage_zip_and_extract(ice_zips[0], LOCAL_STAGE_ICE)
_fp = _h5s[0]

print(f"Inspecting: {_fp.name}")
print(f"Matched TC: {tc_zip_for_ice[ice_zips[0]].name if tc_zip_for_ice[ice_zips[0]] else 'None'}")

with eck.read_product(str(_fp)) as ds:
    print("\n── Dataset overview ──────────────────────────────")
    print(ds)

    print("\n── IWC and r_eff variables ───────────────────────")
    for var, short in DATA_VARS.items():
        err_var = var + "_error"
        for v in [var, err_var]:
            if v in ds.data_vars:
                arr    = ds[v].values.astype(float)
                finite = arr[np.isfinite(arr)]
                zeros  = int((finite == 0).sum())
                print(f"  {v:35s}  shape={arr.shape}  "
                      f"min={np.nanmin(arr):.3g}  max={np.nanmax(arr):.3g}  "
                      f"NaN={100*(1-len(finite)/arr.size):.1f}%  "
                      f"zeros={zeros} ({100*zeros/max(len(finite),1):.1f}% of finite)")
            else:
                print(f"  {v:35s}  *** NOT FOUND ***")

cleanup_staged_data(_local_zip, _extract_dir)
print("\nInspection done.")

## 4. Grid setup and accumulators

Three arrays per variable accumulate pixel statistics in a single pass:

| Accumulator | Summed quantity | Used for |
|---|---|---|
| `sum`    | pixel values           | mean = sum / count |
| `sum_sq` | squared pixel values   | std = sqrt(sum_sq/count − mean²) |
| `count`  | number of valid pixels | denominator; cells < MIN_SAMPLES → NaN |

In [ ]:
def make_accumulators():
    acc = {}
    for var in DATA_VARS:
        acc[var] = {
            "sum":    np.zeros((n_lat, n_lon, n_h), dtype=np.float64),
            "sum_sq": np.zeros((n_lat, n_lon, n_h), dtype=np.float64),
            "count":  np.zeros((n_lat, n_lon, n_h), dtype=np.int64),
        }
    counters = {
        "total_pixels":        0,   # all ice-masked pixels encountered
        "rejected_nonpositive": 0,   # rejected because value <= 0 (fill value)
        "accepted":            0,   # pixels accumulated
    }
    return acc, counters


def merge_acc(global_acc, global_counters, local_acc, local_counters):
    for var in DATA_VARS:
        global_acc[var]["sum"]    += local_acc[var]["sum"]
        global_acc[var]["sum_sq"] += local_acc[var]["sum_sq"]
        global_acc[var]["count"]  += local_acc[var]["count"]
    for k in global_counters:
        global_counters[k] += local_counters[k]


print(f"Accumulator shape per variable : ({n_lat}, {n_lon}, {n_h})")
print(f"Memory per variable (3 arrays) : "
      f"{3 * n_lat * n_lon * n_h * 8 / 1e6:.1f} MB")
print(f"Total for {len(DATA_VARS)} variables         : "
      f"{3 * len(DATA_VARS) * n_lat * n_lon * n_h * 8 / 1e6:.1f} MB")

## 5. Core processing functions

1. **`_build_ice_mask`** — reads ATL_TC__2A and returns a boolean array
   `(n_obs, n_h)` where True = ice cloud at that (profile, height).
2. **`accumulate_one_file`** — for one ATL_ICE_2A HDF5 file: reads data,
   builds the ice mask, interpolates onto `target_h`, applies the `d > 0`
   fill-value filter, then accumulates into grid cells via `np.bincount`.

In [ ]:
def _build_ice_mask(tc_fp, n_obs):
    """
    Build a boolean ice mask from a single ATL_TC__2A HDF5 file.

    Returns
    -------
    ice_mask : bool array, shape (n_obs, n_h)
        True  → pixel is classified as ice cloud AND passes quality checks.
        False → not ice, bad quality, or no valid retrieval.
    """
    ice_mask = np.zeros((n_obs, n_h), dtype=bool)

    with eck.read_product(str(tc_fp)) as ds_tc:
        tc_cls = ds_tc[TC_CLASS_VAR].values.astype(float)
        tc_h   = ds_tc[HEIGHT_VAR].values.astype(float)
        has_qc = TC_QC_VAR in ds_tc.data_vars
        tc_qc  = ds_tc[TC_QC_VAR].values.astype(int) if has_qc else None

    for i in range(min(n_obs, tc_cls.shape[0])):
        h_i   = tc_h[i, :]
        cls_i = tc_cls[i, :]
        qc_ok = (tc_qc[i, :] <= TC_MAX_QC_FLAG) if has_qc else True

        valid = (
            np.isfinite(h_i) & np.isfinite(cls_i)
            & ~np.isin(cls_i, TC_EXCLUDE_CODES)
            & qc_ok
        )
        if valid.sum() < 2:
            continue

        h_v, cls_v = h_i[valid], cls_i[valid]
        order = np.argsort(h_v)
        h_v, cls_v = h_v[order], cls_v[order]
        h_v, idx = np.unique(h_v, return_index=True)
        cls_v = cls_v[idx]
        if h_v.size < 2:
            continue

        # Nearest-neighbour: keeps discrete class codes (no blending)
        ice_v = (cls_v == TC_ICE_CODE).astype(float)
        f = interp1d(h_v, ice_v, kind="nearest", bounds_error=False, fill_value=0.0)
        ice_mask[i, :] = f(target_h).astype(bool)

    return ice_mask


def accumulate_one_file(ice_fp, tc_fp, acc, counters):
    """
    Process one ATL_ICE_2A HDF5 file and add its valid pixels into acc.
    """
    # ── Load data ─────────────────────────────────────────────────────────────
    with eck.read_product(str(ice_fp)) as ds:
        h_raw    = ds[HEIGHT_VAR].values.astype(float)   # (n_obs, n_levels)
        lat      = ds["latitude"].values
        lon      = ds["longitude"].values
        var_data = {v: ds[v].values.astype(float) for v in DATA_VARS}
        n_obs = h_raw.shape[0]

    # ── Ice mask from ATL_TC ──────────────────────────────────────────────────
    ice_mask = _build_ice_mask(tc_fp, n_obs)   # (n_obs, n_h)

    # ── Map lat/lon to grid-cell indices ──────────────────────────────────────
    lat_idx = np.clip(np.searchsorted(lat_bins[1:-1], lat), 0, n_lat - 1)
    lon_idx = np.clip(np.searchsorted(lon_bins[1:-1], lon), 0, n_lon - 1)

    # ── Per-variable accumulation ─────────────────────────────────────────────
    for var in DATA_VARS:
        d_raw = var_data[var]   # (n_obs, n_levels)

        flat_idx_list, val_list, val_sq_list = [], [], []

        for i in range(n_obs):
            # Early exit: skip profiles with no ice at any level
            if not ice_mask[i, :].any():
                continue

            h_i = h_raw[i, :]
            d_i = d_raw[i, :]

            # Interpolation uses all non-NaN raw levels (including fill zeros,
            # which are rejected AFTER interpolation at the accumulation step).
            valid_raw = np.isfinite(h_i) & np.isfinite(d_i)
            if valid_raw.sum() < 2:
                continue

            h_v = h_i[valid_raw]; d_v = d_i[valid_raw]
            order = np.argsort(h_v)
            h_v, d_v = h_v[order], d_v[order]
            h_v, idx_u = np.unique(h_v, return_index=True)
            d_v = d_v[idx_u]
            if h_v.size < 2:
                continue

            d_interp = interp1d(
                h_v, d_v, kind="linear", bounds_error=False, fill_value=np.nan
            )(target_h)

            # Apply ice mask: NaN out non-ice levels
            d_interp[~ice_mask[i, :]] = np.nan

            # Count and reject fill zeros (0.0 is not a valid retrieval)
            is_ice    = ice_mask[i, :]
            is_finite = np.isfinite(d_interp)
            counters["total_pixels"]         += int(is_ice.sum())
            counters["rejected_nonpositive"] += int((is_ice & is_finite & (d_interp <= FILL_VALUE_THRESHOLD)).sum())

            # Keep only positive finite ice pixels
            keep = is_finite & (d_interp > FILL_VALUE_THRESHOLD)
            counters["accepted"] += int(keep.sum())

            valid_h = np.where(keep)[0]
            if len(valid_h) == 0:
                continue

            fi   = lat_idx[i] * n_lon * n_h + lon_idx[i] * n_h + valid_h
            vals = d_interp[valid_h]
            flat_idx_list.append(fi)
            val_list.append(vals)
            val_sq_list.append(vals * vals)

        # Vectorised bincount accumulation
        if flat_idx_list:
            fi_all  = np.concatenate(flat_idx_list).astype(int)
            v_all   = np.concatenate(val_list)
            vsq_all = np.concatenate(val_sq_list)
            flat_size = n_lat * n_lon * n_h

            acc[var]["sum"]    += np.bincount(fi_all, weights=v_all,   minlength=flat_size).reshape(n_lat, n_lon, n_h)
            acc[var]["sum_sq"] += np.bincount(fi_all, weights=vsq_all, minlength=flat_size).reshape(n_lat, n_lon, n_h)
            acc[var]["count"]  += np.bincount(fi_all,                  minlength=flat_size).reshape(n_lat, n_lon, n_h).astype(int)


print("Core functions defined")

## 6. Diagnostic test — 5 ZIPs

Process a small sample before committing to the full batch.

**Expected ranges for December 2025 ice clouds:**
- IWC: bulk of pixels 10⁻³ – 10⁻¹ mg m⁻³; max < 10 mg m⁻³.
- r_eff: bulk 10 – 100 μm; median for thin cirrus ~20–40 μm.

**If stats look wrong:**
- Median or mean still near 0 → `d > 0` filter not taking effect, check `FILL_VALUE_THRESHOLD`.
- Rejection rate > 95% → something upstream is wrong (TC match, HDF5 variable name).
- 0 pixels accepted → verify TC match in Cell 2 and HDF5 variable names in Cell 3.

In [ ]:
N_TEST_ZIPS = 5

test_acc, test_counters = make_accumulators()
test_failed = []

for src_zip in ice_zips[:N_TEST_ZIPS]:
    local_zip = tc_local_zip = extract_dir = tc_extract_dir = None
    try:
        local_zip, extract_dir, h5_files = stage_zip_and_extract(src_zip, LOCAL_STAGE_ICE)
        tc_zip = tc_zip_for_ice.get(src_zip)
        if tc_zip is None:
            print(f"  No TC match for {src_zip.name} — skipping")
            continue
        tc_local_zip, tc_extract_dir, tc_h5s = stage_zip_and_extract(tc_zip, LOCAL_STAGE_TC)
        tc_fp = tc_h5s[0] if tc_h5s else None
        for fp in h5_files:
            if tc_fp is None: continue
            file_acc, file_counters = make_accumulators()
            accumulate_one_file(fp, tc_fp, file_acc, file_counters)
            merge_acc(test_acc, test_counters, file_acc, file_counters)
    except Exception as ex:
        test_failed.append(str(ex))
    finally:
        cleanup_staged_data(local_zip, extract_dir)
        cleanup_staged_data(tc_local_zip, tc_extract_dir)

print(f"Diagnostic test done — {N_TEST_ZIPS} ZIPs, {len(test_failed)} failures\n")

c = test_counters
total = max(c["total_pixels"], 1)
print("── Pixel filter summary ─────────────────────────")
print(f"  Total ice pixels encountered : {c['total_pixels']:>10,}")
print(f"  Rejected (fill zeros, ≤ 0)  : {c['rejected_nonpositive']:>10,}  ({100*c['rejected_nonpositive']/total:.1f}%)")
print(f"  Accepted                     : {c['accepted']:>10,}  ({100*c['accepted']/total:.1f}%)")
print()

print("── Descriptive statistics (gridded cells with count ≥ 1) ───")
for var, short in DATA_VARS.items():
    cnt  = test_acc[var]["count"]
    s    = test_acc[var]["sum"]
    mask = cnt > 0
    if not mask.any():
        print(f"  {short}: no data accumulated")
        continue
    safe_cnt = np.where(mask, cnt, 1)
    means = np.where(mask, s / safe_cnt, np.nan)
    vals  = means[mask]
    p25, p50, p75 = np.nanpercentile(vals, [25, 50, 75])
    print(f"  {short}")
    print(f"    cells with data : {mask.sum():,}")
    print(f"    min / max       : {vals.min():.4g} / {vals.max():.4g}")
    print(f"    mean            : {vals.mean():.4g}")
    print(f"    median          : {p50:.4g}")
    print(f"    IQR             : {p75 - p25:.4g}  (p25={p25:.4g}, p75={p75:.4g})")
    print(f"    p5 / p95        : {np.nanpercentile(vals, 5):.4g} / {np.nanpercentile(vals, 95):.4g}")
    print()

if test_failed:
    print("First failure:", test_failed[0])

## 7. Full batch processing

Process all ZIPs in the date range.  Progress printed every 10 ZIPs.
Accumulators are reset here so this cell is safe to re-run.

In [ ]:
global_acc, global_counters = make_accumulators()
failed = []
n_done = 0

for src_zip in ice_zips:
    local_zip = tc_local_zip = extract_dir = tc_extract_dir = None
    try:
        local_zip, extract_dir, h5_files = stage_zip_and_extract(src_zip, LOCAL_STAGE_ICE)
        tc_zip = tc_zip_for_ice.get(src_zip)
        if tc_zip is None:
            continue
        tc_local_zip, tc_extract_dir, tc_h5s = stage_zip_and_extract(tc_zip, LOCAL_STAGE_TC)
        tc_fp = tc_h5s[0] if tc_h5s else None

        for fp in h5_files:
            if tc_fp is None: continue
            file_acc, file_counters = make_accumulators()
            accumulate_one_file(fp, tc_fp, file_acc, file_counters)
            merge_acc(global_acc, global_counters, file_acc, file_counters)

        n_done += 1
    except Exception as ex:
        failed.append((str(src_zip), str(ex)))
    finally:
        cleanup_staged_data(local_zip, extract_dir)
        cleanup_staged_data(tc_local_zip, tc_extract_dir)

    if n_done % 10 == 0:
        c = global_counters
        print(f"  {n_done}/{len(ice_zips)} ZIPs  |  "
              f"accepted: {c['accepted']:,}  "
              f"rejected fill: {c['rejected_nonpositive']:,}  "
              f"failures: {len(failed)}")

print(f"\nBatch complete: {n_done} ZIPs processed, {len(failed)} failures")
c = global_counters
total = max(c["total_pixels"], 1)
print(f"Accepted {c['accepted']:,} / {c['total_pixels']:,} ice pixels "
      f"({100*c['accepted']/total:.1f}%)")
if failed:
    print("First failure:", failed[0][1])

## 8. Compute statistics

- mean = sum / count
- variance = sum_sq / count − mean²  (computational form)
- std = sqrt(max(variance, 0))  — clamp to avoid tiny negatives from rounding

Cells with < `MIN_SAMPLES` valid pixels are masked to NaN.

In [ ]:
stats = {}

for var, short in DATA_VARS.items():
    cnt = global_acc[var]["count"].astype(float)
    s   = global_acc[var]["sum"]
    s2  = global_acc[var]["sum_sq"]

    has_data = cnt > 0
    safe_cnt = np.where(has_data, cnt, 1)

    mean     = np.where(has_data, s  / safe_cnt, np.nan)
    variance = np.where(has_data, s2 / safe_cnt - mean ** 2, np.nan)
    std      = np.sqrt(np.maximum(variance, 0.0))
    std      = np.where(cnt > 1, std, np.nan)

    mean = np.where(cnt >= MIN_SAMPLES, mean, np.nan)
    std  = np.where(cnt >= MIN_SAMPLES, std,  np.nan)

    stats[var] = {"mean": mean, "std": std, "count": global_acc[var]["count"]}

    valid_cells = int(np.sum(cnt >= MIN_SAMPLES))
    print(f"{short}: {valid_cells:,} cells with ≥{MIN_SAMPLES} samples  |  "
          f"mean range [{np.nanmin(mean):.3g}, {np.nanmax(mean):.3g}]")

print("\nStatistics computed.")

## 9. Save to NetCDF

Two output files:
- **`_3d.nc`** — full lat × lon × height grid (for map plots).
- **`_latheight.nc`** — zonal mean (lat × height, for latitude-height cross-sections).

The zonal mean is computed by summing raw accumulators along the lon axis before
dividing — pixel-count-weighted, not a simple average of cell means.

In [ ]:
lat_centres = (lat_bins[:-1] + lat_bins[1:]) / 2
lon_centres = (lon_bins[:-1] + lon_bins[1:]) / 2

global_attrs = {
    "description":            "ATL_ICE_2A ice cloud microphysics, v2 processing",
    "date_range":             f"{START_DATE} to {END_DATE}",
    "grid_resolution_deg":    GRID_RES_DEG,
    "min_samples":            MIN_SAMPLES,
    "fill_value_threshold":   FILL_VALUE_THRESHOLD,
    "pixels_accepted":        int(global_counters["accepted"]),
    "pixels_rejected_fill":   int(global_counters["rejected_nonpositive"]),
}

tag = f"{START_DATE.replace('-', '')}_{END_DATE.replace('-', '')}"

UNITS = {
    "ice_effective_radius": "um",       # 10⁻⁶ m = μm
    "ice_water_content":    "mg m-3",   # 10⁻⁶ kg m⁻³
}

def make_da_3d(data, long_name, units):
    return xr.DataArray(data.astype(np.float32),
                        dims=["latitude", "longitude", "height"],
                        attrs={"long_name": long_name, "units": units})

def make_da_lh(data, long_name, units):
    return xr.DataArray(data.astype(np.float32),
                        dims=["latitude", "height"],
                        attrs={"long_name": long_name, "units": units})

data_vars_3d = {}
data_vars_lh = {}

for var, short in DATA_VARS.items():
    u = UNITS[var]

    data_vars_3d[f"{short}_mean"]  = make_da_3d(stats[var]["mean"],                f"{short} cell mean",            u)
    data_vars_3d[f"{short}_std"]   = make_da_3d(stats[var]["std"],                 f"{short} standard deviation",   u)
    data_vars_3d[f"{short}_count"] = make_da_3d(stats[var]["count"].astype(float), f"{short} pixel count per cell", "1")

    # Zonal mean: sum accumulators along lon BEFORE dividing (pixel-count-weighted)
    lh_sum    = global_acc[var]["sum"].sum(axis=1)
    lh_sum_sq = global_acc[var]["sum_sq"].sum(axis=1)
    lh_count  = global_acc[var]["count"].sum(axis=1).astype(float)

    lh_has      = lh_count > 0
    lh_safe_cnt = np.where(lh_has, lh_count, 1)

    lh_mean = np.where(lh_has, lh_sum    / lh_safe_cnt, np.nan)
    lh_var  = np.where(lh_has, lh_sum_sq / lh_safe_cnt - lh_mean ** 2, np.nan)
    lh_std  = np.sqrt(np.maximum(lh_var, 0.0))

    lh_mean = np.where(lh_count >= MIN_SAMPLES, lh_mean, np.nan)
    lh_std  = np.where(lh_count >= MIN_SAMPLES, lh_std,  np.nan)

    data_vars_lh[f"{short}_mean"]  = make_da_lh(lh_mean,  f"{short} zonal mean",              u)
    data_vars_lh[f"{short}_std"]   = make_da_lh(lh_std,   f"{short} zonal standard deviation", u)
    data_vars_lh[f"{short}_count"] = make_da_lh(lh_count, f"{short} zonal pixel count",        "1")

coords_3d = {
    "latitude":  xr.DataArray(lat_centres, dims=["latitude"],  attrs={"units": "degrees_north"}),
    "longitude": xr.DataArray(lon_centres, dims=["longitude"], attrs={"units": "degrees_east"}),
    "height":    xr.DataArray(target_h,    dims=["height"],    attrs={"units": "m"}),
}
coords_lh = {"latitude": coords_3d["latitude"], "height": coords_3d["height"]}

out_3d = OUTPUT_DIR / f"ATL_ICE_2A_v2_{GRID_RES_DEG}deg_{tag}_3d.nc"
out_lh = OUTPUT_DIR / f"ATL_ICE_2A_v2_{GRID_RES_DEG}deg_{tag}_latheight.nc"

xr.Dataset(data_vars_3d, coords=coords_3d, attrs=global_attrs).to_netcdf(out_3d)
xr.Dataset(data_vars_lh, coords=coords_lh, attrs=global_attrs).to_netcdf(out_lh)

print(f"Saved 3D output        : {out_3d}")
print(f"Saved latheight output : {out_lh}")